# FIAP – Fase 6 | Capítulo 1 – FarmTech Solutions
## Visão Computacional com YOLOv5 (Detecção de Objetos)
**Autor:** Deivisson Gonçalves Lima – **RM565095**  
**Grupo:** 47 (Individual)  
**Notebook:** `DeivissonLima_RM565095_fase6_cap1.ipynb`  
**Período:** 10/09/2025 a 14/10/2025

### Classes escolhidas
- `copo` (Coffee cup)
- `controle` (Remote control)


## 1) Setup inicial (Colab + Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/Fase6/Fase6_Cap1'
DATA_DIR = f"{DRIVE_BASE}/data"
RUNS_DIR = f"{DRIVE_BASE}/runs"
print('DRIVE_BASE =', DRIVE_BASE)
print('DATA_DIR   =', DATA_DIR)
print('RUNS_DIR   =', RUNS_DIR)


## 2) Dependências e YOLOv5

In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt
import torch
print('CUDA disponível?', torch.cuda.is_available())
print('Versão torch:', torch.__version__)


## 3) Baixar dataset do Open Images (FiftyOne)
> **Observação importante (Colab / Python 3.12):** pode ocorrer conflito entre `mongoengine` e `pymongo`.
Se acontecer, use o *hotfix* abaixo e **reinicie o runtime** quando o Colab solicitar.

In [ ]:
# --- Hotfix de compatibilidade para Colab (Python 3.12) ---
%pip -q uninstall -y mongoengine pymongo
%pip -q install "pymongo==4.6.3" "mongoengine==0.29.0" "motor==3.4.0"
%pip -q install "fiftyone==0.25.0" pyyaml
import mongoengine, pymongo
print("mongoengine:", mongoengine.__version__, "| pymongo:", pymongo.version)


In [ ]:
import os, random, yaml
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F
CLASSES = ['Coffee cup', 'Remote control']
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
ds = foz.load_zoo_dataset(
    'open-images-v7', split='train', label_types=['detections'],
    classes=CLASSES, only_matching=True, max_samples=1200, shuffle=True, seed=51,
)
base = ds.filter_labels('detections', F('label').is_in(CLASSES))
import random
def pick_ids_for_class(view, class_name, k, exclude_ids=set(), seed=7):
    v = (view.filter_labels('detections', F('label') == class_name)
            .match(F('detections.detections').length() > 0)
            .shuffle(seed=seed))
    ids = []
    for _id in v.values('id'):
        if _id not in exclude_ids:
            ids.append(_id)
        if len(ids) >= k:
            break
    return ids
ids_A = pick_ids_for_class(base, 'Coffee cup', 40, set(), 11)
ids_B = pick_ids_for_class(base, 'Remote control', 40, set(ids_A), 13)
assert len(ids_A)==40 and len(ids_B)==40, 'Ajuste max_samples: não foi possível 40/40 únicas.'
def split_32_4_4(ids):
    random.Random(99).shuffle(ids)
    return ids[:32], ids[32:36], ids[36:40]
train_A,val_A,test_A = split_32_4_4(ids_A)
train_B,val_B,test_B = split_32_4_4(ids_B)
train_ids = train_A + train_B
val_ids   = val_A   + val_B
test_ids  = test_A  + test_B
train_view = base.select(train_ids).filter_labels('detections', F('label').is_in(CLASSES))
val_view   = base.select(val_ids).filter_labels('detections', F('label').is_in(CLASSES))
test_view  = base.select(test_ids).filter_labels('detections', F('label').is_in(CLASSES))
for split, view in [('train',train_view),('val',val_view),('test',test_view)]:
    view.export(export_dir=DATA_DIR, dataset_type=fo.types.YOLOv5Dataset,
                label_field='detections', split=split, classes=CLASSES)
data_yaml = {'train': f'{DATA_DIR}/train/images','val': f'{DATA_DIR}/val/images','test': f'{DATA_DIR}/test/images','nc':2,'names':['copo','controle']}
import yaml, os
with open(os.path.join(DATA_DIR,'data.yaml'),'w') as f:
    yaml.safe_dump(data_yaml,f,sort_keys=False)
print('Dataset YOLOv5 pronto em', DATA_DIR)


### (Plano B) Fallback sem FiftyOne
Se, mesmo com o hotfix, o FiftyOne continuar falhando, use o **fallback** abaixo para baixar o Open Images V7
e converter de **Pascal VOC → YOLO** com `openimages`. **Descomente e rode** somente se necessário.

In [ ]:
# %pip -q install openimages opencv-python-headless pandas tqdm
# import os, shutil, glob, random, xml.etree.ElementTree as ET
# from tqdm import tqdm
# BASE = f"{DRIVE_BASE}/data"
# for p in ["train/images","train/labels","val/images","val/labels","test/images","test/labels"]:
#     os.makedirs(os.path.join(BASE,p), exist_ok=True)
# from openimages.download import download_images
# CLASSES_OI = {"Coffee cup":"copo", "Remote control":"controle"}
# TARGET_PER_CLASS = 40
# download_images(classes=list(CLASSES_OI.keys()), max_images=TARGET_PER_CLASS,
#                 dataset_dir="/content/oi_temp", annotation_format="pascal",
#                 image_format="original")
# def voc_to_yolo_box(w,h,xmin,ymin,xmax,ymax):
#     xc = (xmin + xmax)/2.0 / w
#     yc = (ymin + ymax)/2.0 / h
#     bw = (xmax - xmin) / float(w)
#     bh = (ymax - ymin) / float(h)
#     return xc,yc,bw,bh
# rng = random.Random(99)
# all_records = []
# for cls_oi, cls_name in CLASSES_OI.items():
#     img_dir = os.path.join("/content/oi_temp", cls_oi)
#     xmls = sorted(glob.glob(os.path.join(img_dir, "*.xml")))
#     imgs = [x.replace(".xml",".jpg") for x in xmls]
#     pairs = [(i,x) for i,x in zip(imgs, xmls) if os.path.exists(i)]
#     rng.shuffle(pairs); pairs = pairs[:TARGET_PER_CLASS]
#     train = pairs[:32]; val=pairs[32:36]; test=pairs[36:40]
#     all_records.append((cls_name, train, val, test))
# def save_split(pairs, split, class_idx):
#     for img_path, xml_path in tqdm(pairs, desc=f"{split}"):
#         tree = ET.parse(xml_path); root = tree.getroot()
#         w = int(root.find('size/width').text); h = int(root.find('size/height').text)
#         img_name = os.path.basename(img_path)
#         lbl_name = os.path.splitext(img_name)[0] + ".txt"
#         out_img = os.path.join(BASE, f"{split}/images", img_name)
#         out_lbl = os.path.join(BASE, f"{split}/labels", lbl_name)
#         import shutil
#         shutil.copyfile(img_path, out_img)
#         lines = []
#         for obj in root.findall('object'):
#             xmin = int(obj.find('bndbox/xmin').text)
#             ymin = int(obj.find('bndbox/ymin').text)
#             xmax = int(obj.find('bndbox/xmax').text)
#             ymax = int(obj.find('bndbox/ymax').text)
#             xc,yc,bw,bh = voc_to_yolo_box(w,h,xmin,ymin,xmax,ymax)
#             lines.append(f"{class_idx} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
#         with open(out_lbl, 'w') as f:
#             f.write("\n".join(lines))
# CLASS_INDEX = {"copo":0, "controle":1}
# for cls_name, train, val, test in all_records:
#     idx = CLASS_INDEX[cls_name]
#     save_split(train, 'train', idx)
#     save_split(val,   'val',   idx)
#     save_split(test,  'test',  idx)
# with open(os.path.join(BASE,'data.yaml'),'w') as f:
#     f.write(f"""train: {BASE}/train/images\nval:   {BASE}/val/images\ntest:  {BASE}/test/images\n\nnc: 2\nnames: [\"copo\",\"controle\"]\n""")
# print('Dataset pronto em formato YOLOv5 em:', BASE)


## 4) Treinamento – 30 épocas

In [ ]:
%cd /content/yolov5
!python train.py --img 640 --batch 16 --epochs 30 --data /content/drive/MyDrive/Fase6/Fase6_Cap1/data/data.yaml --weights yolov5s.pt --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e30 --exist-ok


## 5) Treinamento – 60 épocas

In [ ]:
%cd /content/yolov5
!python train.py --img 640 --batch 16 --epochs 60 --data /content/drive/MyDrive/Fase6/Fase6_Cap1/data/data.yaml --weights yolov5s.pt --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e60 --exist-ok


## 6) Validação e Inferência

In [ ]:
%cd /content/yolov5
!python val.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e30/weights/best.pt --data /content/drive/MyDrive/Fase6/Fase6_Cap1/data/data.yaml --task val --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e30_val --exist-ok
!python val.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60/weights/best.pt --data /content/drive/MyDrive/Fase6/Fase6_Cap1/data/data.yaml --task val --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e60_val --exist-ok
!python detect.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60/weights/best.pt --img 640 --conf 0.25 --source /content/drive/MyDrive/Fase6/Fase6_Cap1/data/test/images --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name infer_test_e60 --exist-ok


## 7) Análise e Conclusões
- Compare mAP@0.5 / mAP@0.5:0.95 (30 vs 60)
- Avalie precision/recall e losses
- Inclua prints de `infer_test_e60`
- Discuta limitações e próximos passos
